# Drug Verification — Model Inspection

This notebook loads and inspects the trained PK/PD neural network from the repository.
It is intended for new group members to explore the model architecture, training data, and performance without needing to retrain.

**Branch:** `feat/jess-suggested-vclScalar`  
**Repo:** [lstrsrmn/drug-verification](https://github.com/lstrsrmn/drug-verification)

### Sections
1. Setup
2. Load Model
3. Training Data
4. Evaluation
5. Diagnostic Plots

---

### How the model is produced

The committed `pk.keras` and `pk.onnx` are produced locally by running:

```bash
pk train                                             # trains on simulated PK/PD data, saves pk.keras
pk export --model-path pk.keras --onnx-out pk.onnx  # exports to ONNX for formal verification
```

Training uses standard MSE loss on simulated patient data (50 patients, 48 timesteps).
The model is a 5→128→64→1 feedforward network with ReLU activations.
Inputs are normalised using a `StandardScaler` fitted on the training split only.
A +0.0001 clamp is added at ONNX export to guarantee non-negative outputs for the `nonNeg` verification property.

For formal verification of the committed model, see `verification.ipynb`.

> **Note:** If you have previously run this notebook in Colab, delete the `drug-verification` directory and re-run Setup to pick up the latest committed model.

## 1. Setup

In [ ]:
import os, sys

# Clone the repo if running in Colab
if 'google.colab' in sys.modules:
    if not os.path.exists('drug-verification'):
        !git clone --branch feat/jess-suggested-vclScalar https://github.com/lstrsrmn/drug-verification.git
    os.chdir('drug-verification')

    if os.path.abspath('src') not in sys.path:
        sys.path.insert(0, os.path.abspath('src'))

    !pip install -q tensorflow scikit-learn matplotlib pandas numpy

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
print('Working directory:', os.getcwd())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from drug_verification.simulation import simulate_cohort
from drug_verification.training import prepare_data, evaluate_model
from drug_verification.types import SimulationConfig
from drug_verification import constants as C

print('TensorFlow:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

## 2. Load Model

Loads the committed `pk.keras` weights directly — no retraining.

In [ ]:
MODEL_PATH = 'pk.keras'

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f'{MODEL_PATH} not found. Run `pk train` locally and commit the file.'
    )

model = tf.keras.models.load_model(MODEL_PATH)
print(f'Loaded model from {MODEL_PATH}')
print()
model.summary()

In [ ]:
SEED         = C.DEFAULT_SEED
NUM_PATIENTS = C.NUM_PATIENTS
TIMESTEPS    = C.TIMESTEPS

np.random.seed(SEED)

cfg = SimulationConfig(num_patients=NUM_PATIENTS, timesteps=TIMESTEPS, seed=SEED)
X, y = simulate_cohort(cfg, seed=SEED)

feature_names = ['C (conc)', 'T (temp)', 'WBC', 'Age', 'Weight']
print(f'Samples:  {X.shape[0]}')
print(f'Features: {X.shape[1]}  (C, T, WBC, Age, Weight)')
print(f'Target:   dose (mg)')
print()
for i, name in enumerate(feature_names):
    print(f'  {name:12s}  min={X[:,i].min():.2f}  max={X[:,i].max():.2f}  mean={X[:,i].mean():.2f}')
print(f'  {\"Dose\":12s}  min={y.min():.2f}  max={y.max():.2f}  mean={y.mean():.2f}')

## 3. Training Data

Reproduces the training data using the same seed as `pk train` so the scaler and train/test splits match exactly.

In [ ]:
X_train, X_test, y_train, y_test, scaler = prepare_data(X, y, seed=SEED)

print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')
print()
print('Scaler mean: ', np.round(scaler.mean_, 4))
print('Scaler std:  ', np.round(scaler.scale_, 4))

## 4. Evaluation

In [ ]:
metrics = evaluate_model(model, X_test, y_test)
print('Test set evaluation')
print(f'  MSE:               {metrics["mse"]:.4f}')
print(f'  MAE:               {metrics["mae"]:.4f}')
print(f'  Max absolute error: {metrics["max_absolute_error"]:.4f} mg  ← worst-case overdose prediction')

## 5. Diagnostic Plots

In [ ]:
y_pred = model.predict(X_test, verbose=0).flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# True vs predicted
ax = axes[0]
ax.scatter(y_test.flatten(), y_pred, alpha=0.3, s=10)
lims = [0, max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=1, label='Ideal')
ax.set_xlabel('True dose (mg)')
ax.set_ylabel('Predicted dose (mg)')
ax.set_title('True vs predicted dose')
ax.legend()
ax.grid(True, alpha=0.3)

# Prediction distribution
ax = axes[1]
ax.hist(y_pred, bins=40, alpha=0.6, label='Predicted', color='steelblue')
ax.hist(y_test.flatten(), bins=40, alpha=0.6, label='True', color='orange')
ax.set_xlabel('Dose (mg)')
ax.set_ylabel('Count')
ax.set_title('Prediction distribution')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

zero_preds = np.sum(y_pred == 0)
print(f'Predictions at zero: {zero_preds} / {len(y_pred)} ({100*zero_preds/len(y_pred):.1f}%)  ← ReLU collapse check')

In [ ]:
residuals = y_pred - y_test.flatten()
feature_names = ['C (conc)', 'T (temp)', 'WBC', 'Age', 'Weight']

# Unscale X_test back to original space for interpretable axes
X_test_raw = scaler.inverse_transform(X_test)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, (ax, name) in enumerate(zip(axes, feature_names)):
    ax.scatter(X_test_raw[:, i], residuals, alpha=0.3, s=8)
    ax.axhline(0, color='red', linestyle='--', linewidth=1)
    ax.set_xlabel(name)
    ax.set_ylabel('Residual (mg)' if i == 0 else '')
    ax.set_title(f'Residuals vs {name}')
    ax.grid(True, alpha=0.3)

plt.suptitle('Residuals by feature — systematic patterns indicate bias', y=1.02)
plt.tight_layout()
plt.show()